In [1]:
#Classification (just column removing, SMOTE, and train/test split)

In [2]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns


import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

from imblearn.over_sampling import SMOTE

from scipy.stats import pointbiserialr
from sklearn.preprocessing import StandardScaler

In [3]:

# filepath = "Data/flightweatherdata.csv"
# df = pd.read_csv(filepath, index_col=0)


# print(df.info())
# print(f"\nShape: {df.shape}")

In [4]:
# # Plotting the percentage of observations that fall under each class
# ax = df["ArrDel15"].value_counts().sort_values().plot(kind="barh", color=["r", "g"])
# totals= []
# for i in ax.patches:
#     totals.append(i.get_width())
# total = sum(totals)
# for i in ax.patches:
#      ax.text(i.get_width()+.3, i.get_y()+.20, 
#      str(round((i.get_width()/total)*100, 2))+'%', 
#      fontsize=10, color='black')
# plt.title("ArrDel15", fontsize=20)
# plt.xlabel("Count", fontsize=14)
# plt.ylabel("Class", fontsize=14)
# plt.show()
# print(df["ArrDel15"].value_counts())
# fig = ax.get_figure()

In [5]:
# # encode categorical data into numeric values
# labelEncoder = LabelEncoder()
# df["Origin"] = labelEncoder.fit_transform(df["Origin"])
# df["Dest"] = labelEncoder.fit_transform(df["Dest"])
# df["airport"] = labelEncoder.fit_transform(df["airport"])
# df["OriginAirportID"] = labelEncoder.fit_transform(df["OriginAirportID"])
# df["DestAirportID"] = labelEncoder.fit_transform(df["DestAirportID"])
# df["FlightDate"] = labelEncoder.fit_transform(df["FlightDate"])
# df["date"] = labelEncoder.fit_transform(df["date"])

# #separate categorical and numeric data
# categorical = ["Year", "Quarter", "Month", "DayofMonth", "Origin", "Dest", "CRSDepTime", "DepTime", "DepDel15", "winddirDegree", "weatherCode", "time", "ArrDel15", "ArrTime", "CRSArrTime", "OriginAirportID", "DestAirportID", "airport", "NewTime"]
# numeric = ["DepDelayMinutes", "windspeedKmph", "precipMM", "visibility", "pressure", "cloudcover", "DewPointF", "WindGustKmph", "tempF", "WindChillF", "humidity", "ArrDelayMinutes"]
# target = df["ArrDel15"]

In [6]:
# pbc = []
# for col in numeric:
#     coeff, pval = pointbiserialr(df[col], target)
#     pbc.append([col, coeff, pval])

# pbc_corr = (pd.DataFrame(pbc, columns=["Feature", "CorrCoeff", "pValue"])
#             .sort_values(by="CorrCoeff", ascending=False)
#             .reset_index(drop=True))
# plt.figure(figsize=(6, 6))
# hm = sns.heatmap(pbc_corr.set_index("Feature")[["CorrCoeff"]],
#                  vmin=-1, vmax=1, annot=True, fmt=".3f", cmap="BrBG")
# hm.set_title("Point-Biserial Correlation with ArrDel15", fontsize=14, pad=12)
# plt.tight_layout()
# plt.show()
# pbc_corr


In [7]:
# spearman = (df[categorical].corr(method="spearman")[["ArrDel15"]]
#             .drop(index="ArrDel15")
#             .sort_values(by="ArrDel15", ascending=False))

# plt.figure(figsize=(6, 6))
# hm = sns.heatmap(spearman, vmin=-1, vmax=1, annot=True, fmt=".4f", cmap="BrBG")
# hm.set_title("Spearman Correlation with ArrDel15", fontsize=14, pad=12)
# plt.tight_layout()
# plt.show()

In [8]:
# # Feature-to-feature correlation to spot multicollinearity (the basis for the drops below)
# feature_df = df.drop(columns=["ArrDel15"])
# corr = feature_df.corr(method="pearson")

# # Heatmap of the full feature-feature correlation matrix
# plt.figure(figsize=(14, 11))
# heatmap = sns.heatmap(corr, vmin=-1, vmax=1, annot=True, fmt=".2f",
#                       cmap="BrBG", square=True, annot_kws={"size": 7})
# heatmap.set_title("Feature-to-Feature Correlation Matrix", fontdict={"fontsize": 18}, pad=16)
# plt.tight_layout()
# plt.show()

# # notes highly correlated pairs (|r| > 0.6) -> keep only one feature from each pair
# threshold = 0.6
# upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
# high_pairs = upper.stack().sort_values(ascending=False).loc[lambda s: s > threshold]

# print(f"Highly correlated feature pairs (|r| > {threshold}):\n")
# for (a, b), v in high_pairs.items():
#     print(f"  {a:15s} <-> {b:15s}  r = {v:.3f}")

# del corr, feature_df, upper, high_pairs

In [9]:
# #run from here

# cols_to_drop = ["FlightDate",
#                 "OriginAirportID",
#                 "DestAirportID",
#                 "NewTime",
#                 "date",
#                 "airport",
#                 "WindChillF",
#                 "Quarter",
#                 "WindGustKmph",
#                 "time",
#                 "DayofMonth",
#                 "CRSArrTime",
#                 "ArrTime",
#                 "DepDel15",
#                 #"ArrDelayMinutes"
#                 "DepTime",
#                 "CRSDepTime",
# ]

# #drop Arrival related columns due to leakage?

# df.drop(columns=cols_to_drop, inplace=True)

# print(f"Dropped: {cols_to_drop}")
# print(f"\nShape: {df.shape}")
# print(df.info())

In [10]:
# X = df.loc[:, df.columns != "ArrDel15"]
# y = np.array(df.loc[:, df.columns == "ArrDel15"]["ArrDel15"])

# model = RandomForestClassifier(n_estimators=100,
#                      criterion="entropy", random_state=42, n_jobs=-1)

# model.fit(X, y)
# joblib.dump(model, "./Data/feature_importance_model.joblib")
# model = joblib.load("./Data/feature_importance_model.joblib")
# importances = model.feature_importances_
# importances

In [11]:
#importances = pd.DataFrame({
#    "Feature": list(X.columns),
#    "Importance": model.feature_importances_
#})
#importances = importances.sort_values(by="Importance", ascending=False)
#importances = importances.set_index("Feature")
#importances
#plt.figure(figsize=(8, 6), dpi=80)
#plt.barh(importances.index, importances.Importance)
#plt.title("Feature Importance Ranking obtained from Random Forest Classifier", fontsize=12)
#plt.xlabel("Importances")
#plt.ylabel("Features")
#del importances
#del model
#del X
#del y

In [12]:
# df.to_csv("Data/flightweather_encoded.csv")

In [13]:
filepath = "Data/flightweather_encoded.csv"
df = pd.read_csv(filepath, index_col=0)

print(df.info())
print(f"\nShape: {df.shape}")

<class 'pandas.DataFrame'>
RangeIndex: 1851436 entries, 0 to 1851435
Data columns (total 17 columns):
 #   Column           Dtype  
---  ------           -----  
 0   Year             int64  
 1   Month            int64  
 2   Origin           int64  
 3   Dest             int64  
 4   DepDelayMinutes  float64
 5   ArrDelayMinutes  float64
 6   ArrDel15         float64
 7   windspeedKmph    int64  
 8   winddirDegree    int64  
 9   weatherCode      int64  
 10  precipMM         float64
 11  visibility       int64  
 12  pressure         int64  
 13  cloudcover       int64  
 14  DewPointF        int64  
 15  tempF            int64  
 16  humidity         int64  
dtypes: float64(4), int64(13)
memory usage: 240.1 MB
None

Shape: (1851436, 17)


In [14]:
# ── One train/test split for BOTH stages ───────────────────────────────
# Features exclude BOTH targets. Labels carry ArrDel15 (classification) and
# ArrDelayMinutes (regression) together, so the split stays row-aligned and
# both stages end up sharing the exact same test rows.
X = df.loc[:, ~df.columns.isin(["ArrDel15", "ArrDelayMinutes"])]
y = df[["ArrDel15", "ArrDelayMinutes"]]
feature_cols = X.columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y["ArrDel15"]
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"{len(feature_cols)} features: {feature_cols.to_list()}")
print("\nTrain ArrDel15 balance (pre-SMOTE):")
print(y_train["ArrDel15"].value_counts())

X_train: (1481148, 15) | X_test: (370288, 15)
15 features: ['Year', 'Month', 'Origin', 'Dest', 'DepDelayMinutes', 'windspeedKmph', 'winddirDegree', 'weatherCode', 'precipMM', 'visibility', 'pressure', 'cloudcover', 'DewPointF', 'tempF', 'humidity']

Train ArrDel15 balance (pre-SMOTE):
ArrDel15
0.0    1170702
1.0     310446
Name: count, dtype: int64


In [15]:
# ── SMOTE on the TRAINING set ONLY (test set is never resampled) ───────
# SMOTE resamples against a single target (ArrDel15). To keep the regression
# label aligned with the synthetic rows, append ArrDelayMinutes to the matrix,
# resample by ArrDel15, then peel it back off afterwards.
train_matrix = X_train.copy()
train_matrix["ArrDelayMinutes"] = y_train["ArrDelayMinutes"].to_numpy()

smote = SMOTE(random_state=42)
train_resampled, arrdel15_resampled = smote.fit_resample(
    train_matrix, y_train["ArrDel15"].to_numpy()
)

print(f"Training rows: {train_matrix.shape[0]} -> {train_resampled.shape[0]} after SMOTE")
print(pd.Series(arrdel15_resampled, name="ArrDel15").value_counts())

Training rows: 1481148 -> 2341404 after SMOTE
ArrDel15
0.0    1170702
1.0    1170702
Name: count, dtype: int64


In [16]:
# ── Two variations of the (resampled) training data ───────────────────
# Same feature columns in both; only the label differs. The test halves come
# straight from the untouched split above, so Stage 1 and Stage 2 evaluate on
# identical, aligned, real (non-synthetic) rows.

# Stage 1 — classification (label = ArrDel15)
clf_features_train = train_resampled[feature_cols]
clf_labels_train   = np.asarray(arrdel15_resampled)
clf_features_test  = X_test
clf_labels_test    = np.asarray(y_test["ArrDel15"])

# Stage 2 — regression (label = ArrDelayMinutes)
reg_features_train = train_resampled[feature_cols]
reg_labels_train   = np.asarray(train_resampled["ArrDelayMinutes"])
reg_features_test  = X_test
reg_labels_test    = np.asarray(y_test["ArrDelayMinutes"])

print("clf_features_train:", clf_features_train.shape, "| reg_features_train:", reg_features_train.shape)
print("clf_features_test :", clf_features_test.shape,  "| reg_features_test :", reg_features_test.shape)
print("features:", clf_features_train.columns.to_list())

clf_features_train: (2341404, 15) | reg_features_train: (2341404, 15)
clf_features_test : (370288, 15) | reg_features_test : (370288, 15)
features: ['Year', 'Month', 'Origin', 'Dest', 'DepDelayMinutes', 'windspeedKmph', 'winddirDegree', 'weatherCode', 'precipMM', 'visibility', 'pressure', 'cloudcover', 'DewPointF', 'tempF', 'humidity']


In [17]:
# ── Save Stage 1 (classification) splits ──────────────────────────────
clf_features_train.to_csv("Data/Classification3/clf_features_train.csv")
clf_features_test.to_csv("Data/Classification3/clf_features_test.csv")
pd.Series(clf_labels_train, name="ArrDel15").to_csv("Data/Classification3/clf_labels_train.csv", index=False)
pd.Series(clf_labels_test,  name="ArrDel15").to_csv("Data/Classification3/clf_labels_test.csv",  index=False)

# Scaled versions for models that need it (e.g. Logistic Regression).
# Scaler is fit on TRAIN only, then applied to test.
# scaler = StandardScaler()
# clf_features_train_scaled = scaler.fit_transform(clf_features_train)
# clf_features_test_scaled  = scaler.transform(clf_features_test)

In [30]:
# ── Save Stage 2 (regression) splits ──────────────────────────────────
reg_features_train.to_csv("Data/Regression3/reg_features_train.csv")
reg_features_test.to_csv("Data/Regression3/reg_features_test.csv")
pd.Series(reg_labels_train, name="ArrDelayMinutes").to_csv("Data/Regression3/reg_labels_train.csv", index=False)
pd.Series(reg_labels_test,  name="ArrDelayMinutes").to_csv("Data/Regression3/reg_labels_test.csv",  index=False)

In [19]:
# #Training data
# ax = pd.DataFrame(clf_labels_train).value_counts().sort_values().plot(kind="barh", color=["r", "g"])
# ax.set_axisbelow(True)
# ax.grid()
# totals= []
# for i in ax.patches:
#     totals.append(i.get_width())
# total = sum(totals)
# for i in ax.patches:
#      ax.text(i.get_width()+.3, i.get_y()+.20, 
#      str(round((i.get_width()/total)*100, 2))+'%', 
#      fontsize=10, color='black')
# plt.title("ArrDel15", fontsize=20)
# plt.xlabel("Count", fontsize=14)
# plt.ylabel("Class", fontsize=14)
# plt.show()
# print(pd.DataFrame(clf_labels_train).value_counts())
# fig = ax.get_figure()

In [20]:
# #Logistic Regression

# model = LogisticRegression(n_jobs=-1, max_iter=1000)
# model.fit(clf_features_train_scaled, clf_labels_train)
# joblib.dump(model, "./Data/LogisticRegression.joblib")
# model = joblib.load("./Data/LogisticRegression.joblib")
# model_pred = model.predict(clf_features_test_scaled)
# #print(confusion_matrix(labels_test, model_pred))
# print(classification_report(clf_labels_test, model_pred))
# conf_mat_plot = ConfusionMatrixDisplay.from_estimator(model, clf_features_test_scaled, clf_labels_test)
# plt.title("Logistic Regression")
# plt.show()
# del model
# del model_pred

In [21]:
# #Logistic Regression not scaled

# model = LogisticRegression(n_jobs=-1, max_iter=1000)
# model.fit(clf_features_train, clf_labels_train)
# model_pred = model.predict(clf_features_test)
# print(classification_report(clf_labels_test, model_pred))
# conf_mat_plot = ConfusionMatrixDisplay.from_estimator(model, clf_features_test, clf_labels_test)
# plt.title("Logistic Regression")
# plt.show()
# del model
# del model_pred

In [22]:
# #Decision Tree Classifier

# model = DecisionTreeClassifier()
# model.fit(clf_features_train, clf_labels_train)
# joblib.dump(model, "./Data/DecisionTreeClassifier.joblib")
# model = joblib.load("./Data/DecisionTreeClassifier.joblib")
# model_pred = model.predict(clf_features_test)
# # print(confusion_matrix(labels_test, model_pred))
# print(classification_report(clf_labels_test, model_pred))
# conf_mat_plot = ConfusionMatrixDisplay.from_estimator(model, clf_features_test, clf_labels_test)
# plt.title("Decision Tree Classifier")
# plt.show()
# del model
# del model_pred

In [23]:
# from xgboost import XGBClassifier

# model = XGBClassifier(n_jobs=-1, eval_metric="logloss", random_state=42)
# model.fit(clf_features_train, clf_labels_train)
# joblib.dump(model, "Data/XGBClassifier.joblib")
# model = joblib.load("Data/XGBClassifier.joblib")
# model_pred = model.predict(clf_features_test)
# print(classification_report(clf_labels_test, model_pred))
# conf_mat_plot = ConfusionMatrixDisplay.from_estimator(model, clf_features_test, clf_labels_test)
# plt.title("XGBoost Classifier")
# plt.show()
# del model
# del model_pred

In [24]:
# #Random Forest

# model = RandomForestClassifier(n_jobs=-1)
# model.fit(clf_features_train, clf_labels_train)
# joblib.dump(model, "./Data/RandomForestClassifier.joblib")
# model = joblib.load("./Data/RandomForestClassifier.joblib")
# model_pred = model.predict(clf_features_test)
# # print(confusion_matrix(labels_test, model_pred))
# print(classification_report(clf_labels_test, model_pred))
# conf_mat_plot = ConfusionMatrixDisplay.from_estimator(model, clf_features_test, clf_labels_test)
# plt.title("Random Forest Classifier")
# plt.show()
# del model
# del model_pred

In [25]:
# #Extra Trees

# model = ExtraTreesClassifier(n_jobs=-1)
# model.fit(clf_features_train, clf_labels_train)
# joblib.dump(model, "./Data/ExtraTreesClassifier.joblib")
# model = joblib.load("./Data/ExtraTreesClassifier.joblib")
# model_pred = model.predict(clf_features_test)
# # print(confusion_matrix(labels_test, model_pred))
# print(classification_report(clf_labels_test, model_pred))
# conf_mat_plot = ConfusionMatrixDisplay.from_estimator(model, clf_features_test, clf_labels_test)
# plt.title("Extra Trees Classifier")
# plt.show()

In [26]:
# # Setup — collect rows, then build the DataFrame once
# rows = []

# classifiers = ["DecisionTreeClassifier", "XGBClassifier", "RandomForestClassifier", "ExtraTreesClassifier"]
# for clf in classifiers:
#     model = joblib.load(f"./Data/{clf}.joblib")
#     model_pred2 = model.predict_proba(clf_features_test)[:, 1]
#     fpr, tpr, _ = roc_curve(clf_labels_test, model_pred2)
#     auc = roc_auc_score(clf_labels_test, model_pred2)
#     rows.append({"classifiers": clf,
#                  "fpr": fpr,
#                  "tpr": tpr,
#                  "auc": auc})
#     del model
#     del model_pred2

# for clf in ["LogisticRegression"]:          # note: list, not a bare string
#     model = joblib.load(f"./Data/{clf}.joblib")
#     model_pred3 = model.predict_proba(clf_features_test_scaled)[:, 1]
#     fpr, tpr, _ = roc_curve(clf_labels_test, model_pred3)
#     auc = roc_auc_score(clf_labels_test, model_pred3)
#     rows.append({"classifiers": clf,
#                  "fpr": fpr,
#                  "tpr": tpr,
#                  "auc": auc})
#     del model
#     del model_pred3

# # Build the DataFrame and set the index
# perf_df = pd.DataFrame(rows, columns=["classifiers", "fpr", "tpr", "auc"])
# perf_df.set_index("classifiers", inplace=True)

In [27]:
# fig = plt.figure(figsize=(8,6), dpi=80)
# for clf_name in perf_df.index:
#     plt.plot(perf_df.loc[clf_name]["fpr"], 
#              perf_df.loc[clf_name]["tpr"], 
#              label="{}, AUC={:.3f}".format(clf_name, perf_df.loc[clf_name]["auc"]))
    
# plt.plot([0,1], [0,1], color='orange', linestyle='--')

# plt.xticks(np.arange(0.0, 1.1, step=0.1))
# plt.xlabel("False Positive Rate", fontsize=15)

# plt.yticks(np.arange(0.0, 1.1, step=0.1))
# plt.ylabel("True Positive Rate", fontsize=15)

# plt.title("ROC AUC Analysis", fontweight="bold", fontsize=15)
# plt.legend(prop={"size":13}, loc="lower right")

# plt.show()

In [28]:
# model_specs = {
#     "LogisticRegression":     features_test_scaled,
#     "DecisionTreeClassifier": features_test,
#     "XGBClassifier":          features_test,
#     "RandomForestClassifier": features_test,
#     "ExtraTreesClassifier":   features_test,
# }

# rows = []
# for name, X_test in model_specs.items():
#     model = joblib.load(f"./Data/{name}.joblib")
#     pred = model.predict(X_test)
#     rows.append({
#         "Model":     name,
#         "Accuracy":  accuracy_score(labels_test, pred),
#         "Precision": precision_score(labels_test, pred, pos_label=1),
#         "Recall":    recall_score(labels_test, pred, pos_label=1),
#         "F1":        f1_score(labels_test, pred, pos_label=1),
#     })
#     del model, pred

# metrics_df = (pd.DataFrame(rows)
#               .set_index("Model")
#               .sort_values("F1", ascending=False)
#               .round(4))

# metrics_df


In [29]:
# metrics_df.plot(kind="bar", figsize=(10, 6), rot=15)
# plt.title("Classifier Performance Comparison", fontweight="bold")
# plt.ylabel("Score")
# plt.ylim(0, 1)
# plt.legend(loc="lower right")
# plt.tight_layout()
# plt.show()
